# S&P 500 Pair Trade Finder

End-to-end workflow: constituents → prices → correlation matrix → spread z-scores → ranked pair candidates.

Run the cells top to bottom. Tune the knobs in the config cell below.

Requires: `pip install requests pandas numpy matplotlib lxml`

In [ ]:
# ---- config: tweak these ----
CORR_THRESHOLD = 0.80   # min pairwise correlation to consider a pair
Z_LOOKBACK     = 60     # trading days used for spread mean/std
MIN_HISTORY    = 120    # min shared trading days required
TOP_N          = 15      # how many candidates to show
WORKERS        = 12      # download threads

In [ ]:
import io
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

UA = {"User-Agent": "Mozilla/5.0"}
pd.set_option("display.max_rows", 30)

## 1. S&P 500 constituents

Ticker list + GICS sector from Wikipedia (fresh every run).

In [ ]:
def get_constituents():
    api = "https://en.wikipedia.org/w/api.php"
    r = requests.get(
        api,
        params={"action": "parse", "page": "List_of_S&P 500_companies",
                "format": "json", "prop": "text"},
        headers=UA, timeout=60,
    ).json()
    table = pd.read_html(io.StringIO(r["parse"]["text"]["*"]), header=0)[0]
    table = table[["Symbol", "GICS Sector"]].dropna()
    tickers = table["Symbol"].str.replace(".", "-", regex=False).tolist()  # BRK.B -> BRK-B
    sectors = dict(zip(tickers, table["GICS Sector"]))
    return tickers, sectors


tickers, sectors = get_constituents()
print(f"{len(tickers)} constituents")

## 2. Download 1y of daily prices

Yahoo Finance chart API (no API key, no crumb needed), threaded. Uses adjusted closes.

In [ ]:
def fetch_close(ticker, tries=3):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    for attempt in range(tries):
        try:
            r = requests.get(url, params={"interval": "1d", "range": "1y"},
                             headers=UA, timeout=30)
            if r.status_code == 429:
                time.sleep(5)
                continue
            r.raise_for_status()
            res = r.json()["chart"]["result"][0]
            q = res["indicators"]["quote"][0]
            px = res["indicators"].get("adjclose", [{}])[0].get("adjclose") or q["close"]
            idx = (pd.to_datetime(res["timestamp"], unit="s", utc=True)
                     .tz_convert("America/Chicago").tz_localize(None).normalize())
            s = pd.Series(px, index=idx, name=ticker).dropna()
            return ticker, s if len(s) > 200 else None
        except Exception:
            time.sleep(1 + attempt)
    return ticker, None


closes = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(fetch_close, t): t for t in tickers}
    for i, f in enumerate(as_completed(futs), 1):
        t, s = f.result()
        if s is not None:
            closes[t] = s
        if i % 100 == 0:
            print(f"{i}/{len(tickers)} done, {len(closes)} ok")

closes = pd.DataFrame(closes).sort_index()
print("closes shape:", closes.shape)  # rows = trading days, cols = tickers

## 3. Correlation matrix of daily returns

In [ ]:
rets = closes.pct_change().dropna(how="all")
rets = rets.loc[:, rets.notna().sum() > 200]
corr = rets.corr()
tickers = list(corr.columns)

off = np.triu_indices(len(tickers), 1)
print(f"{len(tickers)} stocks, mean pairwise correlation: {corr.values[off].mean():.4f}")

## 4. Screen for pair candidates

Keep pairs with correlation ≥ threshold (excluding dual share classes), then z-score the
log-price spread against its trailing window. Large |z| = stretched relationship = candidate.

In [ ]:
DUAL_CLASS = {tuple(sorted(p)) for p in
              [("GOOGL", "GOOG"), ("FOX", "FOXA"), ("NWSA", "NWS"), ("UAA", "UA")]}

In [ ]:
def half_life(spread):
    """Mean-reversion half-life in days via OU regression."""
    y = spread.diff().dropna().values
    x = spread.shift(1).dropna().values
    beta = np.polyfit(x, y, 1)[0]
    return -np.log(2) / beta if beta < 0 else np.nan


logp = np.log(closes)
vals = corr.values
n = len(tickers)
rows = []
for i in range(n):
    for j in range(i + 1, n):
        c = vals[i, j]
        if c < CORR_THRESHOLD:
            continue
        a, b = tickers[i], tickers[j]
        if tuple(sorted((a, b))) in DUAL_CLASS:
            continue
        ix = logp[a].dropna().index.intersection(logp[b].dropna().index)
        if len(ix) < MIN_HISTORY:
            continue
        spread = (logp[a].loc[ix] - logp[b].loc[ix]).dropna()
        if len(spread) < MIN_HISTORY:
            continue
        mu, sd = spread.iloc[-Z_LOOKBACK:].mean(), spread.iloc[-Z_LOOKBACK:].std()
        z = (spread.iloc[-1] - mu) / sd if sd > 0 else 0
        r1m_a = closes[a].iloc[-1] / closes[a].iloc[-21] - 1
        r1m_b = closes[b].iloc[-1] / closes[b].iloc[-21] - 1
        rows.append({
            "pair": f"{a}/{b}",
            "corr": round(c, 3),
            "z": round(z, 2),
            "1m_a": round(r1m_a * 100, 1),
            "1m_b": round(r1m_b * 100, 1),
            "half_life_d": round(half_life(spread), 1),
            "trade": f"long {b}, short {a}" if z > 0 else f"long {a}, short {b}",
        })

pairs = (pd.DataFrame(rows)
           .sort_values("z", key=np.abs, ascending=False)
           .reset_index(drop=True))
print(f"{len(pairs)} candidate pairs")
pairs.head(TOP_N)

## 5. Visualize the top spreads

Normalized prices for the top 3 candidates — you want to see the legs snap back together historically.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, (_, row) in zip(axes, pairs.head(3).iterrows()):
    a, b = row["pair"].split("/")
    norm = closes[[a, b]].dropna() / closes[[a, b]].dropna().iloc[0]
    norm.plot(ax=ax, title=f"{row['pair']}  (corr={row['corr']}, z={row['z']:+.2f})", lw=1.2)
    ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

## 6. Reading the output

- **corr** — correlation of daily returns over the trailing year. Higher = tighter relationship.
- **z** — how stretched the spread is vs its trailing 60-day norm. |z| > 2 is interesting, > 3 is rare.
- **half_life_d** — estimated days for half the divergence to decay, from an OU fit. Shorter = faster expected snap-back.
- **trade** — long the laggard, short the leader.

Caveats before sizing anything: a wide z-score is only a trade if the relationship is intact —
check for earnings, guidance cuts, or M&A on either leg. Pairs diverging on fundamentals are
repriced, not mispriced. Dollar-neutralize and watch the spread, not the outrights.